In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "analysis_results"
RESULTS_DIR.mkdir(exist_ok=True)

# Set this to a specific manual ROI folder when you do not want the latest run.
# Example: ROI_ROOT_OVERRIDE = DATA_DIR / "<lif_name>" / "<series_name>" / "manual_rois" / "roi_YYYYMMDD_HHMMSS"
ROI_ROOT_OVERRIDE = None
MAX_OBJECTS_TO_SHOW = 12


def find_latest_roi_root(data_dir: Path = DATA_DIR) -> Path:
    roi_roots = [p for p in data_dir.glob("*/*/manual_rois/roi_*") if p.is_dir()]
    if not roi_roots:
        raise FileNotFoundError(f"No ROI folders found under {data_dir}/<lif>/<series>/manual_rois/roi_*")
    return max(roi_roots, key=lambda p: p.stat().st_mtime)


def read_tiff_stack(path: Path) -> np.ndarray:
    stack = tifffile.imread(path)
    if stack.ndim != 3:
        raise ValueError(f"Expected a 3-D TIFF stack: {path}")
    return stack


def mip(stack: np.ndarray) -> np.ndarray:
    # MATLAB wrote one XY plane per TIFF page, so tifffile reads stacks as Z, Y, X.
    return np.max(stack, axis=0)


def normalize_for_imshow(image: np.ndarray, low: float = 1, high: float = 99.8) -> np.ndarray:
    image = image.astype(np.float32, copy=False)
    lo, hi = np.percentile(image, [low, high])
    if hi <= lo:
        return np.zeros_like(image, dtype=np.float32)
    return np.clip((image - lo) / (hi - lo), 0, 1)


def make_overlay(reference_mip: np.ndarray, target_mip: np.ndarray) -> np.ndarray:
    ref = normalize_for_imshow(reference_mip)
    tgt = normalize_for_imshow(target_mip)
    overlay = np.zeros((*ref.shape, 3), dtype=np.float32)
    overlay[..., 0] = tgt
    overlay[..., 1] = ref
    return overlay


ROI_ROOT = Path(ROI_ROOT_OVERRIDE) if ROI_ROOT_OVERRIDE is not None else find_latest_roi_root()
OBJECT_DIRS = sorted([p for p in ROI_ROOT.glob("object_*") if p.is_dir()])
if not OBJECT_DIRS:
    raise FileNotFoundError(f"No object folders found in {ROI_ROOT}")

manifest_path = ROI_ROOT / "roi_manifest.csv"
roi_manifest = pd.read_csv(manifest_path) if manifest_path.exists() else pd.DataFrame()

print(f"ROI_ROOT: {ROI_ROOT}")
print(f"Objects found: {len(OBJECT_DIRS)}")
if manifest_path.exists():
    display(roi_manifest)

show_dirs = OBJECT_DIRS[:MAX_OBJECTS_TO_SHOW]
fig, axes = plt.subplots(len(show_dirs), 4, figsize=(14, 3.4 * len(show_dirs)), squeeze=False)

for row, object_dir in enumerate(show_dirs):
    reference_mip = mip(read_tiff_stack(object_dir / "reference_crop.tif"))
    target_mip = mip(read_tiff_stack(object_dir / "target_crop.tif"))
    reference_mask_mip = mip(read_tiff_stack(object_dir / "reference_object_mask.tif")) > 0
    target_mask_mip = mip(read_tiff_stack(object_dir / "target_object_mask.tif")) > 0

    mask_overlay = np.zeros((*reference_mask_mip.shape, 3), dtype=np.float32)
    mask_overlay[..., 1] = reference_mask_mip
    mask_overlay[..., 0] = target_mask_mip
    mask_overlay[..., 2] = target_mask_mip

    panels = [
        (normalize_for_imshow(reference_mip), "Reference MIP", "gray"),
        (normalize_for_imshow(target_mip), "Target MIP", "gray"),
        (make_overlay(reference_mip, target_mip), "Target red / Reference green", None),
        (mask_overlay, "Mask MIP", None),
    ]

    for col, (image, title, cmap) in enumerate(panels):
        ax = axes[row, col]
        ax.imshow(image, cmap=cmap)
        ax.set_title(f"{object_dir.name} | {title}", fontsize=10)
        ax.axis("off")

plt.tight_layout()
plt.show()


# Signal Intensity Analysis

목적: registered TIFF stack에서 channel별 signal intensity를 정량화합니다.

이 노트북은 아직 알고리즘을 작성하기 전의 scaffold입니다. Input/output contract와 TODO를 먼저 고정해두고, 이후 구현 셀을 채워 넣습니다.


## 0. Setup


In [ ]:
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "analysis_results"
RESULTS_DIR.mkdir(exist_ok=True)

CHANNELS = ("DAPI", "Reference", "Target")
IMPLEMENTED = False

print({"project_dir": str(PROJECT_DIR), "data_dir": str(DATA_DIR), "results_dir": str(RESULTS_DIR)})


## 1. Input / Output Contract


In [ ]:
INPUTS = {
    "raw_stacks": "data/<lif_name>/<series_name>/stacks/*_stack.tif",
    "registered_stacks": "data/<lif_name>/<series_name>/registered_stacks/*_stack_registered.tif",
    "metadata": "data/<lif_name>/<series_name>/metadata.json",
}

OUTPUTS = []
print("Inputs:")
for key, value in INPUTS.items():
    print(f"  {key}: {value}")


## 2. Algorithm TODO


In [ ]:
TODO = [
    'ROI/object mask 정의',
    'background subtraction 정책 결정',
    'channel별 mean/sum/max/integrated density 계산',
    'series-level summary 생성',
]

for i, item in enumerate(TODO, 1):
    print(f'{i}. {item}')


## 3. Discovery Helpers


In [ ]:
def discover_series(data_dir: Path = DATA_DIR):
    rows = []
    if not data_dir.exists():
        return pd.DataFrame(rows)
    for stacks_dir in sorted(data_dir.glob("*/*/stacks")):
        series_dir = stacks_dir.parent
        rows.append({
            "lif_name": series_dir.parent.name,
            "series_name": series_dir.name,
            "series_dir": series_dir,
            "stacks_dir": stacks_dir,
            "registered_stacks_dir": series_dir / "registered_stacks",
            "metadata_path": series_dir / "metadata.json",
        })
    return pd.DataFrame(rows)


series_df = discover_series()
display(series_df)


## 4. Implementation Area


In [ ]:
if not IMPLEMENTED:
    print("[SKIP] Algorithm is not implemented yet. Fill this notebook before enabling execution in the driver.")
else:
    raise NotImplementedError("Implement the analysis pipeline here.")


## 5. Outputs


In [ ]:
EXPECTED_OUTPUTS = [
    'signal_intensity_summary.csv',
    'per-object/per-volume intensity table',
    'intensity distribution plots',
]

for path in EXPECTED_OUTPUTS:
    print(path)
